# 05. Stage 3: Re-ranking (Maximal Marginal Relevance)

Notebook này xây dựng tầng đa dạng hóa danh sách đề xuất (Re-ranking) bằng giải thuật MMR để tối ưu hóa trải nghiệm người dùng, tránh sự trùng lặp thể loại quá mức.

---

### Phân tích Quyết định Thiết kế:
*   **Tại sao chọn Maximal Marginal Relevance (MMR)?**
    *   Các mô hình xếp hạng độ chính xác (như LightGBM) có xu hướng gợi ý một danh sách toàn các phim rất tương đồng nhau (ví dụ: 10 phim Hành động siêu anh hùng liên tiếp) vì chúng đều có điểm số cao. Điều này dễ gây nhàm chán. **MMR** cân bằng toán học giữa độ liên quan (score) và sự khác biệt (1 - similarity với các phim đã chọn trước đó trong danh sách). Lập trình viên dễ dàng điều chỉnh độ đa dạng qua siêu tham số lambda.
*   **Tại sao không chọn Deterministic Greedy Reranking?**
    *   Greedy thuần túy không có tham số để tinh chỉnh linh hoạt độ đa dạng và khó kết hợp trọng số điểm số gốc từ Ranker.
*   **Tại sao không chọn DPP (Determinant Point Processes)?**
    *   DPP là giải thuật tối ưu hóa xác suất rất mạnh nhưng độ phức tạp tính toán rất cao ($O(K^3)$), khó cài đặt hơn nhiều so với MMR ($O(K^2)$) vốn đơn giản, trực quan và chạy cực nhanh trên CPU.


### Bước 1: Khởi tạo và Vector hóa TF-IDF phục vụ MMR
Tế bào này chuẩn bị dữ liệu văn bản mở rộng gồm thể loại, đạo diễn và diễn viên chính gộp lại làm `mmr_soup`. Sau đó, ta dùng `TfidfVectorizer` để chuyển đổi toàn bộ soup phim sang ma trận TF-IDF biểu diễn đặc trưng nội dung để phục vụ việc tính toán khoảng cách cosine giữa các bộ phim ứng viên.


In [ ]:
import os
import pandas as pd
import numpy as np
import pickle
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer

# Load data và tạo TF-IDF matrix MMR rộng để đo độ đa dạng
movies_df = pd.read_csv(os.path.join("..", "..", "data", "crawler", "movies_crawled.csv"))
movies_df['genres'] = movies_df['genres'].fillna('')
movies_df['director'] = movies_df['director'].fillna('')
movies_df['cast'] = movies_df['cast'].fillna('')

# Xây dựng soup đa dạng hóa mở rộng (Genre + Director + Cast)
movies_df['mmr_soup'] = movies_df.apply(
    lambda r: f"{r['genres'].replace('|', ' ')} {r['director'].replace(' ', '')} {' '.join(r['cast'].split('|')[:3])}", 
    axis=1
)
mmr_vectorizer = TfidfVectorizer(stop_words='english')
tfidf_matrix = mmr_vectorizer.fit_transform(movies_df['mmr_soup'])


### Bước 2: Định nghĩa Thuật toán Đa dạng hóa MMR

#### Nguyên lý thuật toán MMR (Maximal Marginal Relevance):
Để tránh tình trạng danh sách gợi ý bị trùng lặp thể loại quá mức (ví dụ gợi ý liên tiếp 10 phim hành động siêu anh hùng), ta áp dụng thuật toán MMR để cân bằng giữa Độ liên quan (Relevance) và Độ đa dạng (Diversity).
Công thức lựa chọn phần tử tiếp theo $D_i$ đưa vào danh sách đề xuất $S$:
$$\text{MMR} = \arg\max_{D_i \in R \setminus S} \left[ \lambda \cdot \text{Sim}_1(D_i, Q) - (1 - \lambda) \cdot \max_{D_j \in S} \text{Sim}_2(D_i, D_j) \right]$$
Trong đó:
*   $R$ là tập hợp các ứng viên ban đầu (được xếp hạng bởi LightGBM).
*   $S$ là tập hợp các phim đã được lựa chọn vào danh sách gợi ý cuối cùng.
*   $\text{Sim}_1(D_i, Q)$ là điểm số độ liên quan của ứng viên $D_i$ với user (chính là điểm số dự đoán của LightGBM).
*   $\text{Sim}_2(D_i, D_j)$ là độ tương đồng nội dung giữa ứng viên $D_i$ với phim $D_j$ đã được chọn trong danh sách (tính bằng Cosine Similarity trên ma trận TF-IDF).
*   $\lambda \in [0, 1]$ là tham số điều hòa. Nếu $\lambda = 1.0$, hệ thống chỉ quan tâm độ liên quan (xếp hạng gốc). Nếu $\lambda = 0.0$, hệ thống chỉ quan tâm đa dạng hóa tối đa (chọn phim khác biệt nhất).

#### So sánh các thuật toán Đa dạng hóa (Reranking):
| Thuật toán | Nguyên lý | Ưu điểm | Nhược điểm |
| :--- | :--- | :--- | :--- |
| **Deterministic Greedy** | Chọn phim tiếp theo sao cho khác thể loại với phim liền trước | Rất nhanh, cài đặt đơn giản. | Không linh hoạt, không đo được độ tương đồng ngữ nghĩa chi tiết. |
| **MMR** (Lựa chọn) | Cân bằng tuyến tính giữa điểm xếp hạng và cosine similarity với toàn bộ danh sách đã chọn | Hiệu quả cao, có tham số $\lambda$ điều chỉnh linh hoạt. | Chi phí tính toán tăng dần theo kích thước danh sách chọn $O(K^2)$. |
| **DPP** (Determinant Point Processes)| Mô hình hóa danh sách dưới dạng ma trận Kernel và tính định thức | Tối ưu hóa xác suất toàn cục rất chuẩn xác. | Rất khó cài đặt, chi phí tính toán cực kỳ lớn $O(K^3)$. |

Tế bào này định nghĩa hàm `maximal_marginal_relevance` thực hiện duyệt qua các ứng viên chưa chọn, tính điểm phạt đa dạng và trích chọn phim tối ưu MMR.


In [ ]:
# 1. Định nghĩa giải thuật MMR
def maximal_marginal_relevance(item_scores, tfidf_matrix, lambda_param=0.7, top_k=10):
    if not item_scores:
        return []
        
    movie_id_to_idx = {row['movieId']: idx for idx, row in movies_df.iterrows()}
    
    candidates = [item[0] for item in item_scores]
    scores = np.array([item[1] for item in item_scores])
    
    if scores.max() != scores.min():
        scores_norm = (scores - scores.min()) / (scores.max() - scores.min())
    else:
        scores_norm = np.ones_like(scores)
        
    selected_items = []
    unselected_indices = list(range(len(candidates)))
    
    first_choice = np.argmax(scores_norm)
    selected_items.append(candidates[first_choice])
    unselected_indices.remove(first_choice)
    
    while len(selected_items) < top_k and unselected_indices:
        best_mmr = -1
        best_candidate_idx = -1
        
        selected_matrix_indices = [movie_id_to_idx[mid] for mid in selected_items]
        selected_vectors = tfidf_matrix[selected_matrix_indices]
        
        for idx in unselected_indices:
            candidate_id = candidates[idx]
            candidate_matrix_idx = movie_id_to_idx[candidate_id]
            candidate_vector = tfidf_matrix[candidate_matrix_idx]
            
            sim_with_selected = cosine_similarity(candidate_vector, selected_vectors).max()
            
            mmr_val = lambda_param * scores_norm[idx] - (1 - lambda_param) * sim_with_selected
            
            if mmr_val > best_mmr:
                best_mmr = mmr_val
                best_candidate_idx = idx
                
        selected_items.append(candidates[best_candidate_idx])
        unselected_indices.remove(best_candidate_idx)
        
    return selected_items


### Bước 3: Chạy Thử nghiệm MMR đa dạng hóa
Tế bào này chạy thử nghiệm thực tế thuật toán MMR trên một danh sách phim giả định để quan sát sự thay đổi thứ tự và lựa chọn phim trước và sau khi đa dạng hóa.


In [ ]:
# 2. Thử nghiệm MMR đa dạng hóa
test_candidates = [
    (157336, 0.95),   # Interstellar
    (301528, 0.90),   # Toy Story 4
    (83533, 0.88),    # Avatar: Fire and Ash
    (1301310, 0.85),  # Zombies of the Third Reich
    (976912, 0.82),   # Graphic Desires
]

diversified = maximal_marginal_relevance(test_candidates, tfidf_matrix, lambda_param=0.5, top_k=3)
print("Gốc xếp hạng:", [movies_df[movies_df['movieId'] == mid]['title'].values[0] for mid, _ in test_candidates])
print("Sau MMR (Đa dạng):", [movies_df[movies_df['movieId'] == mid]['title'].values[0] for mid in diversified])
